# Vesuvius 2025 - Phase A: Low-Risk Baseline

## Target: 0.55-0.60 Score

**Loss**: `L = Dice + BCE + α × BoundaryLoss`

**α Schedule**: 0.01 → 0.5 (linear, never exceed 0.5)

**Architecture**: 3D ResEnc U-Net (nnU-Net style)

**No**: clDice, Skeleton Recall, Mask3D, Transformers

In [ ]:
# =============================================================================
# CONFIGURATION
# =============================================================================

class CONFIG:
    DATA_DIR = '/kaggle/input/vesuvius-challenge-2025'
    OUTPUT_DIR = '/kaggle/working/phase_a'
    
    EPOCHS = 300
    BATCH_SIZE = 2
    PATCH_SIZE = (128, 128, 128)
    NUM_WORKERS = 4
    
    # α scheduling: 0.01 → 0.5 over 200 epochs
    ALPHA_START = 0.01
    ALPHA_END = 0.5
    ALPHA_RAMP = 200
    
    FILTERS = [32, 64, 128, 256, 320]
    DEEP_SUP = True
    
    LR = 1e-2
    WEIGHT_DECAY = 3e-5
    
    DEVICE = 'cuda'
    AMP = True
    
    FOLD = 0
    SEED = 42

print(f"Phase A: Dice + BCE + α×BoundaryLoss")
print(f"α: {CONFIG.ALPHA_START} → {CONFIG.ALPHA_END}")

In [ ]:
# =============================================================================
# IMPORTS
# =============================================================================

import os, json, random, gc
import numpy as np
import pandas as pd
from PIL import Image, ImageSequence
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

from sklearn.model_selection import GroupKFold
from scipy.ndimage import distance_transform_edt

def set_seed(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)

set_seed(CONFIG.SEED)
os.makedirs(CONFIG.OUTPUT_DIR, exist_ok=True)

print(f"PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# =============================================================================
# DATA UTILITIES
# =============================================================================

def load_tif(path):
    img = Image.open(path)
    return np.stack([np.array(p) for p in ImageSequence.Iterator(img)], axis=0)

def normalize(vol):
    p01, p99 = np.percentile(vol, [1, 99])
    vol = np.clip(vol, p01, p99)
    return ((vol - vol.mean()) / (vol.std() + 1e-8)).astype(np.float32)

def compute_sdf(mask):
    """Signed distance: negative inside, positive outside."""
    fg = (mask == 1).astype(np.uint8)
    if fg.sum() == 0:
        return np.ones_like(mask, dtype=np.float32)
    if fg.sum() == fg.size:
        return -np.ones_like(mask, dtype=np.float32)
    din = distance_transform_edt(fg)
    dout = distance_transform_edt(1 - fg)
    sdf = dout - din
    return (sdf / max(np.abs(sdf).max(), 1.0)).astype(np.float32)

print("Utilities defined.")

In [ ]:
# =============================================================================
# AUGMENTATION
# =============================================================================

class Aug3D:
    def __init__(self, p_flip=0.5, p_rot=0.3, p_noise=0.15):
        self.p_flip = p_flip
        self.p_rot = p_rot
        self.p_noise = p_noise
    
    def __call__(self, img, mask, sdf):
        for ax in [0, 1, 2]:
            if random.random() < self.p_flip:
                img = np.flip(img, ax).copy()
                mask = np.flip(mask, ax).copy()
                sdf = np.flip(sdf, ax).copy()
        
        if random.random() < self.p_rot:
            k = random.randint(1, 3)
            axes = random.choice([(0,1), (0,2), (1,2)])
            img = np.rot90(img, k, axes).copy()
            mask = np.rot90(mask, k, axes).copy()
            sdf = np.rot90(sdf, k, axes).copy()
        
        if random.random() < self.p_noise:
            img = img + np.random.normal(0, 0.05, img.shape).astype(np.float32)
        
        return img, mask, sdf

print("Augmentation defined.")

In [ ]:
# =============================================================================
# DATASET
# =============================================================================

class VesuviusDS(Dataset):
    def __init__(self, imgs, lbls, patch_size=(128,128,128), ppv=4, aug=True):
        self.imgs = imgs
        self.lbls = lbls
        self.ps = patch_size
        self.ppv = ppv
        self.aug = Aug3D() if aug else None
    
    def __len__(self):
        return len(self.imgs) * self.ppv
    
    def _patch(self, img, lbl):
        d, h, w = img.shape
        pd, ph, pw = [min(s, dim) for s, dim in zip(self.ps, img.shape)]
        
        fg = lbl == 1
        if random.random() < 0.6 and fg.sum() > 0:
            c = np.argwhere(fg)[random.randint(0, fg.sum()-1)]
            z0, y0, x0 = [max(0, min(c[i] - self.ps[i]//2, dim - self.ps[i])) 
                          for i, dim in enumerate([d, h, w])]
        else:
            z0, y0, x0 = [random.randint(0, max(0, dim - self.ps[i])) 
                          for i, dim in enumerate([d, h, w])]
        
        ip = img[z0:z0+pd, y0:y0+ph, x0:x0+pw]
        lp = lbl[z0:z0+pd, y0:y0+ph, x0:x0+pw]
        
        if ip.shape != self.ps:
            pad = [(0, self.ps[i] - ip.shape[i]) for i in range(3)]
            ip = np.pad(ip, pad)
            lp = np.pad(lp, pad, constant_values=2)
        
        return ip, lp
    
    def __getitem__(self, idx):
        vi = idx // self.ppv
        img = normalize(load_tif(self.imgs[vi]))
        lbl = load_tif(self.lbls[vi])
        
        ip, lp = self._patch(img, lbl)
        mask = (lp == 1).astype(np.float32)
        ignore = (lp == 2).astype(np.float32)
        sdf = compute_sdf(lp)
        
        if self.aug:
            ip, mask, sdf = self.aug(ip, mask, sdf)
        
        return {
            'image': torch.from_numpy(ip).float().unsqueeze(0),
            'mask': torch.from_numpy(mask).float().unsqueeze(0),
            'sdf': torch.from_numpy(sdf).float().unsqueeze(0),
            'ignore': torch.from_numpy(ignore).float().unsqueeze(0)
        }

print("Dataset defined.")

In [ ]:
# =============================================================================
# MODEL - 3D ResEnc U-Net
# =============================================================================

class CB(nn.Module):
    def __init__(self, ic, oc):
        super().__init__()
        self.c = nn.Sequential(
            nn.Conv3d(ic, oc, 3, 1, 1, bias=False), nn.InstanceNorm3d(oc, affine=True), nn.LeakyReLU(0.01, True),
            nn.Conv3d(oc, oc, 3, 1, 1, bias=False), nn.InstanceNorm3d(oc, affine=True), nn.LeakyReLU(0.01, True))
        self.s = nn.Conv3d(ic, oc, 1) if ic != oc else nn.Identity()
    def forward(self, x): return self.c(x) + self.s(x)

class UNet3D(nn.Module):
    def __init__(self, ic=1, nc=2, f=[32,64,128,256,320], ds=True):
        super().__init__()
        self.ds = ds
        self.enc = nn.ModuleList()
        self.pool = nn.ModuleList()
        for i, fi in enumerate(f):
            self.enc.append(CB(ic if i==0 else f[i-1], fi))
            if i < len(f)-1:
                self.pool.append(nn.Conv3d(fi, fi, 2, 2))
        
        self.up = nn.ModuleList()
        self.dec = nn.ModuleList()
        for i in range(len(f)-2, -1, -1):
            self.up.append(nn.ConvTranspose3d(f[i+1], f[i], 2, 2))
            self.dec.append(CB(f[i]*2, f[i]))
        
        self.out = nn.Conv3d(f[0], nc, 1)
        if ds:
            self.dout = nn.ModuleList([nn.Conv3d(f[i], nc, 1) for i in range(1, len(f)-1)])
    
    def forward(self, x):
        sk = []
        for i, (e, p) in enumerate(zip(self.enc[:-1], self.pool)):
            x = e(x); sk.append(x); x = p(x)
        x = self.enc[-1](x)
        
        do = []
        for i, (u, d) in enumerate(zip(self.up, self.dec)):
            x = u(x)
            s = sk[-(i+1)]
            if x.shape[2:] != s.shape[2:]:
                x = F.interpolate(x, s.shape[2:], mode='trilinear', align_corners=False)
            x = d(torch.cat([x, s], 1))
            if self.ds and i < len(self.dout):
                do.append(self.dout[i](x))
        
        return {'output': self.out(x), 'deep': do}

m = UNet3D(f=CONFIG.FILTERS, ds=CONFIG.DEEP_SUP)
print(f"Params: {sum(p.numel() for p in m.parameters() if p.requires_grad):,}")
del m; gc.collect()

In [ ]:
# =============================================================================
# LOSS: Dice + BCE + α×BoundaryLoss
# =============================================================================

class DiceLoss(nn.Module):
    def forward(self, pred, target, valid=None):
        ps = torch.softmax(pred, 1)[:, 1:2]
        if valid is not None:
            ps, target = ps * valid, target * valid
        inter = (ps * target).sum()
        union = ps.sum() + target.sum()
        return 1.0 - (2*inter + 1e-5) / (union + 1e-5)

class BoundaryLoss(nn.Module):
    """L_B = mean(pred_fg * sdf). Penalizes predictions outside boundary."""
    def forward(self, pred, sdf, valid=None):
        ps = torch.softmax(pred, 1)[:, 1:2]
        loss = ps * sdf
        if valid is not None:
            loss = loss * valid
            return loss.sum() / (valid.sum() + 1e-8)
        return loss.mean()

class PhaseALoss(nn.Module):
    def __init__(self, a_start=0.01, a_end=0.5, a_ramp=200):
        super().__init__()
        self.a_start, self.a_end, self.a_ramp = a_start, a_end, a_ramp
        self.dice = DiceLoss()
        self.bce = nn.CrossEntropyLoss(reduction='none')
        self.bnd = BoundaryLoss()
        self.dw = [0.5, 0.25, 0.125]
    
    def get_alpha(self, e):
        if e >= self.a_ramp: return self.a_end
        return self.a_start + (self.a_end - self.a_start) * e / self.a_ramp
    
    def forward(self, out, target, sdf, ignore, epoch):
        pred = out['output']
        valid = 1.0 - ignore
        alpha = self.get_alpha(epoch)
        
        # Dice
        dl = self.dice(pred, target, valid)
        
        # BCE
        bce_raw = self.bce(pred, target.squeeze(1).long())
        bl = (bce_raw * valid.squeeze(1)).sum() / (valid.sum() + 1e-8)
        
        # Boundary
        bnl = self.bnd(pred, sdf, valid)
        
        total = dl + bl + alpha * bnl
        
        # Deep supervision
        for i, dp in enumerate(out['deep']):
            if i >= len(self.dw): break
            sz = dp.shape[2:]
            tr = F.interpolate(target, sz, mode='nearest')
            sr = F.interpolate(sdf, sz, mode='trilinear', align_corners=False)
            vr = F.interpolate(valid, sz, mode='nearest')
            total += self.dw[i] * (self.dice(dp, tr, vr) + alpha * self.bnd(dp, sr, vr))
        
        return {'loss': total, 'dice': dl, 'bce': bl, 'boundary': bnl, 'alpha': alpha}

crit = PhaseALoss(CONFIG.ALPHA_START, CONFIG.ALPHA_END, CONFIG.ALPHA_RAMP)
print("α Schedule:")
for e in [0, 50, 100, 150, 200, 250, 300]:
    print(f"  Epoch {e:3d}: α = {crit.get_alpha(e):.3f}")

In [ ]:
# =============================================================================
# METRICS & CHECKPOINT
# =============================================================================

@torch.no_grad()
def dice_score(pred, target):
    pb = (torch.softmax(pred, 1)[:, 1] > 0.5).float()
    tb = target.squeeze(1).float()
    return ((2*(pb*tb).sum() + 1e-5) / (pb.sum() + tb.sum() + 1e-5)).item()

class CkptMgr:
    def __init__(self, path):
        self.path = path
        os.makedirs(path, exist_ok=True)
        self.best = -1
        self.best_ep = -1
    
    def save(self, model, opt, sched, scaler, ep, met, is_best=False):
        ckpt = {'epoch': ep, 'model': model.state_dict(), 'optimizer': opt.state_dict(),
                'scheduler': sched.state_dict(), 'scaler': scaler.state_dict(),
                'metrics': met, 'best': self.best, 'best_ep': self.best_ep}
        torch.save(ckpt, f"{self.path}/last.pth")
        if is_best:
            torch.save(ckpt, f"{self.path}/best.pth")
            print(f"  💾 Best: {met['val_dice']:.4f}")
    
    def load(self, model, opt=None, sched=None, scaler=None, p=None):
        p = p or f"{self.path}/last.pth"
        if not os.path.exists(p): return 0
        c = torch.load(p, map_location=CONFIG.DEVICE)
        model.load_state_dict(c['model'])
        if opt: opt.load_state_dict(c['optimizer'])
        if sched: sched.load_state_dict(c['scheduler'])
        if scaler: scaler.load_state_dict(c['scaler'])
        self.best, self.best_ep = c['best'], c['best_ep']
        print(f"Loaded ep {c['epoch']}, best {self.best:.4f}")
        return c['epoch'] + 1
    
    def update(self, d, ep):
        if d > self.best:
            self.best, self.best_ep = d, ep
            return True
        return False

print("Checkpoint manager defined.")

In [ ]:
# =============================================================================
# TRAINING LOOP
# =============================================================================

def train_ep(model, loader, crit, opt, scaler, dev, ep):
    model.train()
    tl, td, tb, n = 0, 0, 0, 0
    for b in tqdm(loader, desc=f'Ep{ep} Train', leave=False):
        img, mask, sdf, ign = [b[k].to(dev) for k in ['image','mask','sdf','ignore']]
        opt.zero_grad()
        with autocast(enabled=CONFIG.AMP):
            out = model(img)
            L = crit(out, mask, sdf, ign, ep)
        scaler.scale(L['loss']).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 12.0)
        scaler.step(opt); scaler.update()
        tl += L['loss'].item(); td += L['dice'].item(); tb += L['boundary'].item(); n += 1
    return {'train_loss': tl/n, 'train_dice': td/n, 'train_bnd': tb/n}

@torch.no_grad()
def val_ep(model, loader, crit, dev, ep):
    model.eval()
    tl, td, n = 0, 0, 0
    for b in tqdm(loader, desc=f'Ep{ep} Val', leave=False):
        img, mask, sdf, ign = [b[k].to(dev) for k in ['image','mask','sdf','ignore']]
        with autocast(enabled=CONFIG.AMP):
            out = model(img)
            L = crit(out, mask, sdf, ign, ep)
        d = dice_score(out['output'], mask)
        tl += L['loss'].item(); td += d; n += 1
    return {'val_loss': tl/n, 'val_dice': td/n}

print("Training loop defined.")

In [ ]:
# =============================================================================
# DATA PREPARATION
# =============================================================================

def prep_data():
    df = pd.read_csv(f"{CONFIG.DATA_DIR}/train.csv")
    imgd, lbld = f"{CONFIG.DATA_DIR}/train_images", f"{CONFIG.DATA_DIR}/train_labels"
    imgs, lbls, scrolls = [], [], []
    for _, r in df.iterrows():
        ip, lp = f"{imgd}/{r['id']}.tif", f"{lbld}/{r['id']}.tif"
        if os.path.exists(ip) and os.path.exists(lp):
            imgs.append(ip); lbls.append(lp); scrolls.append(r['scroll_id'])
    print(f"Found {len(imgs)} samples, {len(set(scrolls))} scrolls")
    return imgs, lbls, scrolls

IMGS, LBLS, SCROLLS = prep_data()

# GroupKFold
gkf = GroupKFold(n_splits=5)
for fold, (tr, va) in enumerate(gkf.split(np.arange(len(SCROLLS)), groups=SCROLLS)):
    if fold == CONFIG.FOLD:
        TRAIN_IDX, VAL_IDX = tr, va
        print(f"Fold {fold}: Train {len(tr)}, Val {len(va)}")
        break

In [ ]:
# =============================================================================
# MAIN TRAINING
# =============================================================================

def train(resume_path=None):
    print(f"\n{'='*60}")
    print(f"PHASE A: Dice + BCE + α×BoundaryLoss")
    print(f"α: {CONFIG.ALPHA_START} → {CONFIG.ALPHA_END}")
    print(f"{'='*60}\n")
    
    # Data
    tr_ds = VesuviusDS([IMGS[i] for i in TRAIN_IDX], [LBLS[i] for i in TRAIN_IDX], CONFIG.PATCH_SIZE, 4, True)
    va_ds = VesuviusDS([IMGS[i] for i in VAL_IDX], [LBLS[i] for i in VAL_IDX], CONFIG.PATCH_SIZE, 2, False)
    tr_dl = DataLoader(tr_ds, CONFIG.BATCH_SIZE, True, num_workers=CONFIG.NUM_WORKERS, pin_memory=True, drop_last=True)
    va_dl = DataLoader(va_ds, CONFIG.BATCH_SIZE, False, num_workers=CONFIG.NUM_WORKERS, pin_memory=True)
    print(f"Train: {len(tr_ds)} patches, Val: {len(va_ds)} patches\n")
    
    # Model
    model = UNet3D(f=CONFIG.FILTERS, ds=CONFIG.DEEP_SUP).to(CONFIG.DEVICE)
    crit = PhaseALoss(CONFIG.ALPHA_START, CONFIG.ALPHA_END, CONFIG.ALPHA_RAMP)
    
    # Optimizer
    opt = torch.optim.SGD(model.parameters(), lr=CONFIG.LR, momentum=0.99, 
                          weight_decay=CONFIG.WEIGHT_DECAY, nesterov=True)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda e: (1 - e/CONFIG.EPOCHS)**0.9)
    scaler = GradScaler(enabled=CONFIG.AMP)
    ckpt = CkptMgr(CONFIG.OUTPUT_DIR)
    
    # Resume
    start = ckpt.load(model, opt, sched, scaler, resume_path) if resume_path else 0
    
    hist = []
    for ep in range(start, CONFIG.EPOCHS):
        tr_m = train_ep(model, tr_dl, crit, opt, scaler, CONFIG.DEVICE, ep)
        va_m = val_ep(model, va_dl, crit, CONFIG.DEVICE, ep)
        sched.step()
        
        lr = sched.get_last_lr()[0]
        alpha = crit.get_alpha(ep)
        
        print(f"\nEp {ep:03d} | LR {lr:.2e} | α {alpha:.3f}")
        print(f"  Train: loss={tr_m['train_loss']:.4f} dice={tr_m['train_dice']:.4f} bnd={tr_m['train_bnd']:.4f}")
        print(f"  Val:   loss={va_m['val_loss']:.4f} dice={va_m['val_dice']:.4f}")
        
        is_best = ckpt.update(va_m['val_dice'], ep)
        ckpt.save(model, opt, sched, scaler, ep, {**tr_m, **va_m, 'lr': lr, 'alpha': alpha}, is_best)
        hist.append({**tr_m, **va_m, 'lr': lr, 'alpha': alpha})
        
        with open(f"{CONFIG.OUTPUT_DIR}/history.json", 'w') as f:
            json.dump(hist, f, indent=2)
        
        torch.cuda.empty_cache()
    
    print(f"\n{'='*60}")
    print(f"Done! Best: {ckpt.best:.4f} @ ep {ckpt.best_ep}")
    return ckpt.best

print("train() defined. Run next cell to start.")

In [ ]:
# =============================================================================
# START TRAINING
# =============================================================================

# Uncomment to start:
# best = train()

In [ ]:
# =============================================================================
# RESUME
# =============================================================================

# Uncomment to resume:
# best = train(f"{CONFIG.OUTPUT_DIR}/last.pth")

In [ ]:
# =============================================================================
# STATUS CHECK
# =============================================================================

def status():
    p = f"{CONFIG.OUTPUT_DIR}/last.pth"
    if os.path.exists(p):
        c = torch.load(p, map_location='cpu')
        print(f"Epoch: {c['epoch']}/{CONFIG.EPOCHS}")
        print(f"Val Dice: {c['metrics']['val_dice']:.4f}")
        print(f"α: {c['metrics']['alpha']:.3f}")
        print(f"Best: {c['best']:.4f} @ ep {c['best_ep']}")
    else:
        print("No checkpoint.")

status()

## Phase A Summary

### Loss
```
L = Dice + BCE + α × BoundaryLoss
```

### α Schedule
| Epoch | α |
|-------|------|
| 0 | 0.01 |
| 100 | 0.26 |
| 200+ | 0.50 |

### Why
- Early: Learn geometry (Dice + BCE)
- Late: Prevent merges (Boundary Loss)

### Target
Val Dice ≥ 0.55 → Proceed to Phase B

### Checkpoints
- `best.pth` - Best val dice
- `last.pth` - Latest epoch